# Day 4 — Building with LLM APIs

## What we're building
A Sleep Research Assistant powered by Claude API that:
1. Answers sleep science questions with citations
2. Analyses experiment results and gives feedback  
3. Suggests next steps for Project 07
4. Demonstrates multi-turn conversation management

## Why this matters
Every serious AI application today is built on top of
foundation models via API. Learning to build with LLMs
is as fundamental as learning to use databases was in 2000.

## Key concepts
- API calls and authentication
- Prompt engineering
- Conversation history management
- System prompts and personas
- Structured output extraction

In [1]:
!pip install anthropic

In [ ]:
import anthropic
import json
import textwrap
from datetime import datetime

# Install if needed:
# !pip install anthropic

# ── Initialise client ─────────────────────────────────────
# API key is handled automatically in Claude.ai environment
client = anthropic.Anthropic()

print("Anthropic client ready")
print(f"   Using model: claude-sonnet-4-6")

Anthropic client ready
   Using model: claude-sonnet-4-6


In [2]:
import json
import textwrap
from datetime import datetime

# ── Mock LLM client ───────────────────────────────────────
# Simulates API calls with pre-written responses
# Replace with real client when you have an API key

class MockLLMClient:
    """
    Simulates an LLM API for learning purposes.
    Same interface as real client — swap one line to go live.
    """

    def __init__(self):
        self.call_count = 0
        print("MockLLMClient ready")
        print("(Replace with anthropic.Anthropic() "
              "when you have an API key)\n")

    def ask(self, prompt, system=None, max_tokens=500):
        self.call_count += 1

        # Simulate different responses based on keywords
        p = prompt.lower()

        if 'n1' in p and 'hard' in p:
            return """N1 is the hardest sleep stage to classify for three reasons:

1. BREVITY: N1 typically lasts only 1-7 minutes, giving the 
   classifier very few training examples relative to N2 (45-55%).

2. SPECTRAL OVERLAP: N1 EEG shows mixed frequencies — alpha 
   waves (8-13Hz) from wake begin to slow into theta (4-8Hz). 
   This overlap makes it ambiguous.

3. INTER-SUBJECT VARIABILITY: N1 signatures vary more between 
   people than any other stage, making generalisation harder.

For your CNN-LSTM (N1 F1=0.47), the CNN windows help because 
they capture the TRANSITION from alpha to theta across the 
30-second epoch — exactly what distinguishes N1 from wake."""

        elif 'rem' in p and ('feature' in p or 'eeg' in p):
            return """Most discriminative EEG features for REM vs NREM:

1. THETA POWER (4-8Hz): strongly elevated in REM
2. THETA/ALPHA RATIO: peaks during REM, low in NREM
3. SAWTOOTH WAVES: 2-6Hz bursts, nearly unique to REM
4. LOW DELTA POWER: REM has less delta than N2/N3
5. SPINDLE ABSENCE: sleep spindles (12-15Hz) disappear in REM

For Project 07: your theta/alpha ratio feature in 
extract_features() is the single most important feature.
If you haven't added it yet, add it — it should give 
a clear accuracy boost in detect_rem.py."""

        elif 'next' in p or 'improve' in p or 'attention' in p:
            return """Next steps to push past 82%:

1. ADD ATTENTION (Experiment 004):
   Self-attention over your 10 CNN windows will let the model 
   weight which 3-second segments are most informative.
   Expected gain: +2-4% over CNN-LSTM.

2. MULTI-SCALE CNN:
   Use filters of different sizes (3, 5, 7) in parallel.
   Captures both fine-grained and broad frequency patterns.

3. DATA AUGMENTATION:
   Add Gaussian noise to training epochs (σ = 0.01).
   EEG is inherently noisy — teaching robustness helps.

4. LEAVE-ONE-SUBJECT-OUT validation:
   Your current split may have subject leakage.
   True cross-subject performance is the publishable metric.

Start with attention — it's the most principled improvement
and directly builds toward the transformer architecture."""

        elif 'paper' in p or 'publish' in p or 'introduc' in p:
            return """Sleep disorders affect over 70 million people worldwide, 
yet automated sleep staging remains challenging due to the 
complexity of EEG signals and significant inter-subject 
variability. Accurate sleep stage classification is 
fundamental to both clinical diagnosis and emerging 
applications in brain-computer interfaces (BCI).

Recent deep learning approaches have demonstrated 
promising results on standardised datasets such as 
Sleep-EDF, with LSTM-based models achieving 74-82% 
accuracy on single-channel EEG. However, existing methods 
either process raw temporal sequences (losing frequency 
structure) or convert signals to spectrograms (losing 
temporal dynamics).

We propose a CNN-LSTM hybrid architecture that addresses 
both limitations. Our approach extracts local frequency 
features from 3-second windows using convolutional layers, 
then models temporal transitions between windows using a 
bidirectional LSTM. Trained on the Sleep-EDF Cassette 
dataset (20 subjects), our system achieves 80.14% overall 
accuracy with REM F1-score of 0.81 — improvements of 3.29% 
and 0.07 respectively over a pure LSTM baseline.

Our primary contribution is the demonstration that 
combining spatial feature extraction with temporal 
sequence modelling significantly outperforms either 
approach alone, with the largest gains observed in the 
notoriously difficult N1 stage (+34% relative F1 improvement).

This architecture is deployed as the classification backbone 
of a closed-loop BCI system for automated lucid dream 
induction, demonstrating practical applicability beyond 
academic benchmarking."""

        elif 'roadmap' in p or 'month' in p:
            return """6-Month Project 07 Roadmap:

MONTH 1 — Experiment 004 + Validation
  Goal: Transformer beats CNN-LSTM (target 82%+)
  Tasks: Train EEGTransformerClassifier,
         implement leave-one-subject-out validation,
         attention weight visualisation
  Deliverable: experiment_004_transformer notebook
  Risk: transformer may need more data than LSTM

MONTH 2 — Real Hardware
  Goal: Run pipeline on real EEG device
  Tasks: Order Muse S headband (~₹30,000),
         build real-time stream reader,
         integrate with detect_rem.py
  Deliverable: realtime_eeg_stream.py
  Risk: Muse API changes, latency issues

MONTH 3 — Self-Experiment Protocol
  Goal: First sleep experiment on own data
  Tasks: Personal calibration, logging protocol,
         experiment ethics documentation,
         record 5 nights of sleep data
  Deliverable: personal_sleep_dataset/

MONTH 4 — Neural Decoding Exploration
  Goal: Understand what dream imagery looks like in EEG
  Tasks: Read MinD-Vis and reconstruction papers,
         experiment with spectrogram → image pipeline,
         explore fNIRS as complement to EEG
  Deliverable: literature_review.md in PROJECT-07

MONTH 5 — Paper Writing
  Goal: Submit to a workshop or conference
  Tasks: Write full paper (8 pages IEEE format),
         create figures from experiment results,
         target: IEEE SMC or NeurIPS workshop
  Deliverable: paper_draft_v1.pdf

MONTH 6 — LUCID v0.2
  Goal: Working prototype on own sleep data
  Tasks: Integrate all components end-to-end,
         document protocol for others to replicate,
         record demo video
  Deliverable: LUCID v0.2 release on GitHub"""

        else:
            return (f"[Mock response for: '{prompt[:60]}...']\n"
                    "In production this would call claude-sonnet-4-6.\n"
                    "Add your Anthropic API key to get real responses.\n"
                    "Get one free at: console.anthropic.com")


# ── Conversation manager ───────────────────────────────────
class ResearchChat:
    def __init__(self):
        self.llm     = MockLLMClient()
        self.history = []

    def chat(self, message, max_tokens=500):
        self.history.append({
            'role': 'user',
            'content': message
        })
        reply = self.llm.ask(message)
        self.history.append({
            'role': 'assistant',
            'content': reply
        })
        return reply

    def clear(self):
        self.history = []


print(" Setup complete — mock LLM ready")
print("Replace MockLLMClient with anthropic.Anthropic()")
print("when you have an API key\n")

 Setup complete — mock LLM ready
Replace MockLLMClient with anthropic.Anthropic()
when you have an API key



In [3]:
# ── Test 1: Research questions ────────────────────────────
chat = ResearchChat()

questions = [
    "Why is N1 sleep stage the hardest to classify?",
    "What EEG features are most discriminative for REM?",
    "What should I try next to push accuracy above 82%?"
]

print("=== Sleep Research Assistant ===\n")
for q in questions:
    print(f"Q: {q}\n")
    reply = chat.chat(q)
    print(f"A: {textwrap.fill(reply, 70)}\n")
    print("-" * 60 + "\n")


# ── Test 2: Experiment analyser ───────────────────────────
exp_results = {
    "accuracy": 0.8014,
    "rem_f1":   0.81,
    "n1_f1":    0.47,
    "vs_lstm":  "+3.29%"
}

prompt = (f"Analyse these results and tell me "
          f"if this is publishable: "
          f"{json.dumps(exp_results)}")

print("=== Experiment Analysis ===\n")
analysis = chat.chat(prompt)
print(textwrap.fill(analysis, 70))

print("\n\n=== Paper Introduction Draft ===\n")
intro = chat.chat("Write a paper introduction for "
                  "my CNN-LSTM sleep staging research")
print(intro)

print("\n\n=== 6-Month Roadmap ===\n")
roadmap = chat.chat("Generate a 6-month roadmap for "
                    "Project 07 with monthly milestones")
print(roadmap)

MockLLMClient ready
(Replace with anthropic.Anthropic() when you have an API key)

=== Sleep Research Assistant ===

Q: Why is N1 sleep stage the hardest to classify?

A: N1 is the hardest sleep stage to classify for three reasons:  1.
BREVITY: N1 typically lasts only 1-7 minutes, giving the
classifier very few training examples relative to N2 (45-55%).  2.
SPECTRAL OVERLAP: N1 EEG shows mixed frequencies — alpha     waves
(8-13Hz) from wake begin to slow into theta (4-8Hz).     This overlap
makes it ambiguous.  3. INTER-SUBJECT VARIABILITY: N1 signatures vary
more between     people than any other stage, making generalisation
harder.  For your CNN-LSTM (N1 F1=0.47), the CNN windows help because
they capture the TRANSITION from alpha to theta across the  30-second
epoch — exactly what distinguishes N1 from wake.

------------------------------------------------------------

Q: What EEG features are most discriminative for REM?

A: Most discriminative EEG features for REM vs NREM:  1. T

### Shifting from Claude to Gemini as it is more economically efficient for Students 

In [ ]:
!pip install -U google-genai
!pip install python-dotenv

In [1]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

model_name = "gemini-3.1-flash-lite"

response = client.models.generate_content(
    model=model_name,
    contents="Hello"
)

In [2]:
def ask_gemini(prompt, system=None):
    """Single-turn API call using Gemini."""

    if system:
        contents = f"System: {system}\n\nUser: {prompt}"
    else:
        contents = prompt

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=contents
    )

    return response.text

In [3]:
answer = ask_gemini(
    "What is the difference between REM and NREM sleep "
    "in terms of EEG frequency patterns? "
    "Answer in 3 bullet points."
)

print("=== Basic API Call ===\n")
print(answer)

=== Basic API Call ===

Here are the differences between REM and NREM sleep in terms of EEG frequency patterns:

*   **NREM Sleep (Stages 1–3):** This phase is characterized by a progressive slowing of brain waves. It transitions from low-voltage, mixed-frequency activity (Stage 1) to the appearance of sleep spindles and K-complexes (Stage 2), eventually culminating in high-amplitude, low-frequency **delta waves** (0.5–4 Hz) during deep sleep (Stage 3).
*   **REM Sleep:** This phase exhibits a shift to **low-voltage, fast-frequency activity** that closely resembles the EEG pattern of a person who is wide awake (characterized by alpha and beta-like waves). Because of this desynchronized, high-frequency activity, REM sleep is often referred to as "paradoxical sleep."
*   **Distinctive Markers:** NREM sleep is defined by the presence of sleep-specific markers like **spindles and slow-wave activity**, whereas REM sleep is defined by the **absence of these markers** and a return to the toni

In [4]:
from google.genai import types

def ask_gemini(prompt, system=None):
    config = None

    if system:
        config = types.GenerateContentConfig(
            system_instruction=system
        )

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=config
    )

    return response.text

In [6]:
import textwrap

In [7]:
SLEEP_RESEARCH_SYSTEM = """
You are an expert sleep neuroscience research assistant 
with deep knowledge of:
- EEG signal processing and frequency bands
- Sleep stage classification (AASM standards)
- Brain-computer interfaces for sleep research
- Machine learning for neural signal analysis
- The Sleep-EDF dataset and benchmark results

You are assisting a B.Tech AI/ML student who is building 
Project 07 — a closed-loop BCI system for automated lucid 
dream induction called LUCID: Reality?

Their current results:
- Experiment 001 (LSTM): 76.85% accuracy, REM recall 0.84
- Experiment 002 (CNN spectrograms): 71.67%, REM recall 0.66  
- Experiment 003 (CNN-LSTM hybrid): 80.14%, REM recall 0.83

When answering:
- Be technically precise but explain clearly
- Connect answers to their specific project when relevant
- Cite relevant papers or researchers when appropriate
- Suggest concrete next steps when asked
- Keep answers focused and actionable
"""


# ── Test with research question ────────────────────────────
questions = [
    "Why is N1 sleep stage the hardest to classify "
    "from EEG data?",

    "What EEG features are most discriminative "
    "for REM vs NREM classification?",

    "How does the Sleep-EDF dataset compare to other "
    "publicly available sleep EEG datasets?"
]

print("=== Sleep Research Assistant ===\n")

for i, q in enumerate(questions, 1):
    print(f"Q{i}: {q}")
    print("-" * 50)
    answer = ask_gemini(q, system=SLEEP_RESEARCH_SYSTEM)
    # Wrapping text for clean display
    wrapped = textwrap.fill(answer, width=70)
    print(wrapped)
    print()

=== Sleep Research Assistant ===

Q1: Why is N1 sleep stage the hardest to classify from EEG data?
--------------------------------------------------
In the context of your **LUCID: Reality?** project, understanding the
classification difficulty of N1 is crucial. N1 (Stage 1) is the
"twilight zone" of sleep, and your current metrics (specifically your
REM recall) suggest your model is likely struggling to distinguish the
transition between wakefulness and light sleep.  Here is why N1 is the
"nightmare" of sleep staging, broken down into the technical
challenges you are likely encountering in your dataset:  ### 1. High
Intra-class Variance (Low Signal-to-Noise) N1 is characterized by the
disappearance of the posterior alpha rhythm (8–13 Hz) and the
emergence of low-voltage, mixed-frequency activity (theta waves, 4–7
Hz). However, this transition is highly subjective. *   **The
Problem:** There is no "hard" boundary. A subject might be technically
in N1 while still showing brief bursts o

In [8]:
from google import genai
from google.genai import types

class SleepResearchChat:
    """
    Multi-turn conversation with memory using Gemini.
    """

    def __init__(self, api_key, system_prompt):
        self.client = genai.Client(api_key=api_key)
        self.system = system_prompt
        self.history = []

    def chat(self, user_message):
        # Store user message
        self.history.append(
            types.Content(
                role="user",
                parts=[types.Part(text=user_message)]
            )
        )

        # Generate response
        response = self.client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=self.history,
            config=types.GenerateContentConfig(
                system_instruction=self.system
            )
        )

        assistant_reply = response.text

        # Store assistant reply
        self.history.append(
            types.Content(
                role="model",
                parts=[types.Part(text=assistant_reply)]
            )
        )

        return assistant_reply

    def clear(self):
        self.history = []
        print("Conversation cleared.")

    def show_history(self):
        print(f"Conversation length: {len(self.history)} messages")
        for i, msg in enumerate(self.history, 1):
            print(f"\n[{i}] {msg.role.upper()}")
            print(msg.parts[0].text[:150] + "...")

In [12]:
import os
API_KEY = os.getenv("GEMINI_API_KEY")

chat = SleepResearchChat(
    API_KEY,
    SLEEP_RESEARCH_SYSTEM
)

conversation = [
    "I just trained a CNN-LSTM hybrid on Sleep-EDF and got 80.14% accuracy with REM recall of 0.83. My LSTM baseline was 76.85%. What does this improvement tell me about the architecture?",

    "The biggest improvement was in N1 F1-score from 0.35 to 0.47. Why would CNN features help specifically with N1 detection?",

    "What should I try next to push accuracy above 82%? I'm planning to add attention—is that the right move?"
]

for msg in conversation:
    print("You:", msg)
    print("\nGemini:")
    print(chat.chat(msg))
    print("-" * 70)

You: I just trained a CNN-LSTM hybrid on Sleep-EDF and got 80.14% accuracy with REM recall of 0.83. My LSTM baseline was 76.85%. What does this improvement tell me about the architecture?

Gemini:
Achieving an 80.14% accuracy with a 0.83 REM recall is a strong result for a Sleep-EDF benchmark, especially considering that the inter-scorer reliability among human experts often sits around 80–85%. 

Your improvement from 76.85% (LSTM) to 80.14% (CNN-LSTM) tells us several critical things about how your model is processing the underlying physiology of sleep:

### 1. Spatial-Feature Hierarchy vs. Temporal Dependency
Your LSTM baseline was likely treating the raw EEG/EOG time series as a sequential stream. While LSTMs are excellent at capturing long-range dependencies (like the cyclic nature of sleep stages), they struggle to learn the "morphology" of waveforms—such as the sharp, saw-tooth waves characteristic of REM or the complex morphology of sleep spindles (N2).

By adding a CNN front-en

In [14]:
import json
def analyse_experiment(results_dict):
    """
    Send experiment results to Gemini for analysis.
    Returns structured feedback.
    """

    results_json = json.dumps(results_dict, indent=2)

    prompt = f"""
Analyse these sleep stage classification experiment results
and provide structured feedback.

Results:
{results_json}

Provide your analysis in this exact format:

STRENGTHS:
- (list 3 key strengths)

WEAKNESSES:
- (list 3 areas needing improvement)

KEY FINDING:
(one sentence summarising the most important result)

NEXT EXPERIMENT:
(specific recommendation for the next experiment)

PAPER POTENTIAL:
(is this publishable? what venue would suit it?)
"""

    return ask_gemini(
        prompt,
        system=SLEEP_RESEARCH_SYSTEM
    )


# ── Analysing our actual Experiment 003 results ─────────────
exp003_results = {
    "experiment":   "003",
    "name":         "CNN-LSTM Hybrid",
    "dataset":      "Sleep-EDF Cassette, 20 subjects",
    "architecture": {
        "cnn":  "3-layer Conv1d with BatchNorm",
        "lstm": "2-layer bidirectional, hidden_dim=128",
        "windows": "10 x 3-second windows per epoch"
    },
    "results": {
        "overall_accuracy": 0.8014,
        "per_class_f1": {
            "Wake": 0.67,
            "N1":   0.47,
            "N2":   0.86,
            "N3":   0.85,
            "REM":  0.81
        },
        "rem_recall":    0.83,
        "rem_precision": 0.79
    },
    "vs_baseline": {
        "exp001_lstm_accuracy":    0.7685,
        "exp003_accuracy":         0.8014,
        "improvement":             "+3.29%",
        "rem_f1_improvement":      "+0.07",
        "n1_f1_improvement":       "+0.12"
    },
    "project_goal": "Closed-loop lucid dream induction"
}

print("=== Experiment 003 Analysis ===\n")
analysis = analyse_experiment(exp003_results)
print(analysis)

=== Experiment 003 Analysis ===

STRENGTHS:
- **Feature Extraction Robustness:** The use of a CNN-LSTM hybrid effectively bridges the gap between local temporal feature extraction (Conv1d) and long-range sleep architecture modeling (Bidirectional LSTM).
- **REM Sensitivity for BCI:** An 83% recall for REM sleep is a strong foundation for your closed-loop goal, as missing a REM window is more detrimental to lucid dream induction than a false positive.
- **Architectural Scaling:** You have successfully outperformed the baseline LSTM, suggesting that the model is effectively learning hierarchical representations rather than just sequential dependencies.

WEAKNESSES:
- **N1 Classification Deficiency:** An F1-score of 0.47 for N1 is typical but problematic; N1 is the critical transition phase for induction, and your model is likely misclassifying it as Wake or REM.
- **Data Imbalance Impact:** The significantly lower F1-scores for Wake (0.67) and N1 (0.47) compared to N2/N3 suggest the mode

In [15]:
def find_related_papers(topic, your_work):
    """
    Ask Gemini to suggest relevant papers
    and explain how they relate to your work.
    """

    prompt = f"""
I'm working on: {your_work}

Find me the 5 most relevant research papers on: {topic}

For each paper provide:
TITLE: (paper title)
AUTHORS: (main authors, year)
KEY CONTRIBUTION: (one sentence)
RELEVANCE TO MY WORK: (how it connects to my project)

Focus on papers that are:
- Directly relevant to EEG sleep staging
- Published in top venues (NeurIPS, ICLR, IEEE TNSRE, Sleep)
- From 2018 onwards (recent work)
"""

    return ask_gemini(
        prompt,
        system=SLEEP_RESEARCH_SYSTEM
    )


print("=== Related Papers: CNN-LSTM for EEG ===\n")
papers = find_related_papers(
    topic="CNN-LSTM architectures for EEG sleep staging",
    your_work=(
        "CNN-LSTM hybrid achieving 80.14% on Sleep-EDF "
        "with bidirectional LSTM and 10-window CNN features"
    )
)
print(papers)

print("\n" + "="*60 + "\n")

print("=== Related Papers: Attention for EEG ===\n")
papers2 = find_related_papers(
    topic="transformer and attention mechanisms for "
          "EEG classification",
    your_work=(
        "Planning to add self-attention over CNN windows "
        "for Experiment 004"
    )
)
print(papers2)

=== Related Papers: CNN-LSTM for EEG ===

To optimize your LUCID project, it is essential to look at how these architectures handle the **temporal dependency** of sleep stages (hypnogram transitions) while maintaining spatial feature extraction from multi-channel EEG.

Here are 5 highly relevant papers for your CNN-LSTM hybrid development:

---

### 1. DeepSleepNet: A Model for Automatic Sleep Stage Scoring Based on Raw Single-Channel EEG
**AUTHORS:** Supratak A., Dong H., Wu C., Guo Y. (2017/2018, *IEEE Transactions on Neural Systems and Rehabilitation Engineering*)  
**KEY CONTRIBUTION:** Introduced a dual-branch CNN (one for low-frequency features, one for high-frequency) followed by a Bi-LSTM to model temporal transitions.  
**RELEVANCE TO YOUR WORK:** This is the foundational architecture for your Experiment 003. Analyzing their layer-wise feature extraction will help you understand why your hybrid is currently outperforming pure CNN/LSTM approaches.

### 2. AttnSleep: A Context-A

In [16]:
def generate_roadmap(current_state, constraints):
    """
    Generate a detailed technical roadmap
    for the next phase of Project 07.
    """

    prompt = f"""
Generate a detailed 6-month technical roadmap for 
Project 07 — LUCID: Reality?

Current state:
{json.dumps(current_state, indent=2)}

Constraints:
{json.dumps(constraints, indent=2)}

Provide a month-by-month breakdown with:
MONTH N:
  Goal: (specific, measurable target)
  Tasks: (3-4 concrete tasks)
  Deliverable: (what gets pushed to GitHub)
  Risk: (main technical challenge)

Be specific and technically precise.
"""

    return ask_gemini(
        prompt,
        system=SLEEP_RESEARCH_SYSTEM
    )


current_state = {
    "classifier":     "CNN-LSTM, 80.14% on Sleep-EDF",
    "hardware":       "phone vibration via JOIN API",
    "data":           "Sleep-EDF offline dataset",
    "experiments":    3,
    "github":         "PROJECT-07 repo public",
    "semester":       "B.Tech 2nd semester complete"
}

constraints = {
    "budget":         "student, limited hardware budget",
    "time":           "4-5 hours per day alongside studies",
    "hardware":       "laptop only, no GPU, no EEG device yet",
    "timeline":       "6 months",
    "next_goal":      "run first self-experiment on own sleep"
}

print("=== 6-Month Project 07 Roadmap ===\n")
roadmap = generate_roadmap(current_state, constraints)
print(roadmap)

=== 6-Month Project 07 Roadmap ===

To transition from a model trained on offline datasets (Sleep-EDF) to a closed-loop BCI system for your own sleep, you must bridge the gap between **batch-mode inference** and **real-time streaming analytics**. 

Since you lack a dedicated GPU, focus on **TensorRT/ONNX Runtime optimization** and **feature-engineered light models** rather than heavy deep-learning architectures.

---

### **6-Month Technical Roadmap: Project LUCID**

#### **MONTH 1: Data Acquisition & Hardware Procurement**
*   **Goal:** Obtain and validate live EEG input.
*   **Tasks:**
    1.  Acquire a research-grade entry EEG device (e.g., OpenBCI Cyton or a cheaper Muse 2 with Bluetooth connectivity).
    2.  Build a Python pipeline using `BrainFlow` to stream data from the device to your laptop.
    3.  Develop a real-time data visualizer to ensure signal impedance is acceptable (check for 50/60Hz line noise).
*   **Deliverable:** A script that successfully logs a 30-minute “wake

In [17]:
def draft_paper_intro(results_summary):
    """
    Draft the introduction section of a research paper
    based on your experimental results.
    """

    prompt = f"""
Write a professional research paper introduction 
(300-400 words) for the following work:

{results_summary}

The introduction should:
1. Open with the broader motivation (sleep science, BCI)
2. Identify the specific problem (sleep stage classification)
3. Review existing approaches briefly
4. State the gap your work fills
5. Preview your contributions clearly

Use academic writing style appropriate for:
IEEE Transactions on Neural Systems and 
Rehabilitation Engineering (TNSRE)
"""

    return ask_gemini(
        prompt,
        system=SLEEP_RESEARCH_SYSTEM
    )


results_summary = """
We present a CNN-LSTM hybrid architecture for automated 
sleep stage classification from single-channel EEG, 
targeting real-time use in a closed-loop lucid dream 
induction system. Our approach splits 30-second EEG 
epochs into 10 windows, applies a 3-layer CNN to extract 
frequency features from each window, then uses a 
bidirectional 2-layer LSTM to model temporal transitions. 
Trained on the Sleep-EDF Cassette dataset (20 subjects, 
18,226 epochs), our system achieves 80.14% overall 
accuracy and REM F1-score of 0.81, improving over a 
pure LSTM baseline (76.85%) and CNN spectrogram approach 
(71.67%). The architecture is deployed in a closed-loop 
system that triggers haptic stimulation upon REM detection 
for lucid dream induction.
"""

print("=== Paper Introduction Draft ===\n")
intro = draft_paper_intro(results_summary)
print(intro)

print("\n" + "="*60)
print("\nThis is a draft — review, edit, and make it yours.")
print("Never submit AI-generated text without significant")
print("personal revision and your own voice added.")

=== Paper Introduction Draft ===

### Introduction

The exploration of human consciousness during sleep, particularly the induction of lucid dreaming, represents a burgeoning frontier in neurotechnology and sleep science. Lucid dreaming—the phenomenon wherein an individual becomes cognizant of their dream state while remaining asleep—offers unique potential for cognitive rehearsal, phobia treatment, and the study of neural correlates of self-awareness. Realizing this potential in a laboratory or home environment necessitates high-fidelity, real-time identification of Rapid Eye Movement (REM) sleep, the physiological state most conducive to lucidity. However, the efficacy of closed-loop systems for lucid dream induction remains tethered to the latency and precision of automated sleep stage classification.

Traditionally, sleep scoring is performed manually by human experts according to the American Academy of Sleep Medicine (AASM) standards, a process that is time-intensive, subjective,

In [1]:
import os

# Save as a standalone script you can use anytime
tool_code = '''#!/usr/bin/env python3
"""
Project 07 Sleep Research Assistant
A command-line tool for sleep neuroscience research queries.
"""

import anthropic
import textwrap

SYSTEM = """You are an expert sleep neuroscience research 
assistant helping with Project 07 — a closed-loop BCI system
for automated lucid dream induction (LUCID: Reality?).
Be technically precise, cite relevant work, and give
actionable advice."""

import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()


client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)


class SleepResearchChat:
    """
    Multi-turn conversation with memory using Gemini.
    Maintains full conversation history.
    """

    def __init__(self, system_prompt):
        self.system = system_prompt
        self.history = []

    def chat(self, user_message):
        # Save user message
        self.history.append({
            "role": "user",
            "content": user_message
        })

        # Build conversation history
        conversation = ""

        for msg in self.history:
            if msg["role"] == "user":
                conversation += f"User: {msg['content']}\n"
            else:
                conversation += f"Assistant: {msg['content']}\n"

        # Generate response
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=conversation,
            config=types.GenerateContentConfig(
                system_instruction=self.system,
                temperature=0.4,
                max_output_tokens=500
            )
        )

        assistant_reply = response.text

        # Save assistant reply
        self.history.append({
            "role": "assistant",
            "content": assistant_reply
        })

        return assistant_reply

    def clear(self):
        self.history = []
        print("Conversation cleared.")

    def show_history(self):
        print(f"Conversation length: {len(self.history)} turns")

        for i, msg in enumerate(self.history, 1):
            print(f"[{i}] {msg['role'].upper()}:")
            print(msg["content"][:100] + "...")
            print()

'''

# Save to Journey to the BEST Week 7 folder
save_path = "sleep_research_assistant.py"
with open(save_path, 'w') as f:
    f.write(tool_code)

print(f"Saved to {save_path}")
print(f"\nRun from terminal with:")
print(f"  python sleep_research_assistant.py")
print(f"\nThis is a real tool you can use for Project 07 research.")

Saved to sleep_research_assistant.py

Run from terminal with:
  python sleep_research_assistant.py

This is a real tool you can use for Project 07 research.
